# FedX-PALM: Federated Learning Client on Google Colab Pro

**Palm Fruit Ripeness Detection** using YOLOv11 + Federated Learning + Differential Privacy + XAI

---

## Architecture
```
Google Colab Pro (GPU T4/A100)     VPS Server (CPU)
┌──────────────────────────┐       ┌─────────────────────┐
│  Client 1, 2, 3, 4      │──────▶│  FL Server          │
│  YOLOv11 + Local DP     │◀──────│  FedAvg + Central DP│
│  + XAI (Grad-CAM/SHAP)  │       │  Aggregation        │
└──────────────────────────┘       └─────────────────────┘
```

This notebook runs **4 FL clients** sequentially on Colab, each training on its own data partition.

## 1. Setup Environment

In [ ]:
# Install dependencies
!pip install -q ultralytics roboflow flask requests torch torchvision \
    numpy scipy opencv-python matplotlib pyyaml tqdm

In [ ]:
# Verify GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Clone Repository

In [ ]:
import os

# Clone the FedX-PALM repository
if not os.path.exists("fedx-palm"):
    !git clone https://github.com/rachmadiantyy/fedx-palm.git

os.chdir("fedx-palm")
print(f"Working directory: {os.getcwd()}")

## 3. Download & Split Dataset from Roboflow

In [ ]:
from roboflow import Roboflow

# Download Palm Fruit Ripeness Detection dataset
rf = Roboflow(api_key="Ej0bSMpeSri3ky0IYkOU")
project = rf.workspace("dydy-worker").project("palm-fruit-ripeness-detection-f6sac-ccb2z")
version = project.version(2)
dataset = version.download("yolov11", location="./data/raw")

print(f"\nDataset downloaded to: {dataset.location}")

In [ ]:
# Split dataset for 4 FL clients (IID split)
!python scripts/download_dataset.py \
    --skip-download \
    --output-dir ./data/raw \
    --split-clients 4 \
    --split-strategy iid \
    --client-data-dir ./data \
    --seed 42

In [ ]:
# Verify data split
import os
for i in range(1, 5):
    client_dir = f"./data/client_{i}"
    if os.path.exists(f"{client_dir}/images/train"):
        train_imgs = len(os.listdir(f"{client_dir}/images/train"))
        val_imgs = len(os.listdir(f"{client_dir}/images/val")) if os.path.exists(f"{client_dir}/images/val") else 0
        print(f"Client {i}: {train_imgs} train images, {val_imgs} val images")
    else:
        print(f"Client {i}: directory not found")

## 4. Configuration

⚠️ **IMPORTANT**: Update `SERVER_URL` with your VPS IP address!

In [ ]:
# ============================================
# CONFIGURATION - UPDATE THESE VALUES
# ============================================

# Your VPS server IP address (where FL server is running)
SERVER_URL = "http://<YOUR_VPS_IP>:8080"  # <-- CHANGE THIS!

# Model settings
MODEL_VARIANT = "yolo11n.pt"
NUM_CLASSES = 4  # unripe, underripe, ripe, overripe
IMG_SIZE = 640

# Training settings
LOCAL_EPOCHS = 5
BATCH_SIZE = 16
LEARNING_RATE = 0.01

# Differential Privacy settings
DP_ENABLED = True
DP_EPSILON = 1.0
DP_DELTA = 1e-5
DP_MAX_GRAD_NORM = 1.0

# Number of FL rounds to run
MAX_ROUNDS = 100

# Number of clients
NUM_CLIENTS = 4

print(f"Server URL: {SERVER_URL}")
print(f"Model: {MODEL_VARIANT} ({NUM_CLASSES} classes)")
print(f"Training: {LOCAL_EPOCHS} epochs, batch={BATCH_SIZE}, lr={LEARNING_RATE}")
print(f"DP: enabled={DP_ENABLED}, epsilon={DP_EPSILON}")
print(f"Clients: {NUM_CLIENTS}, Rounds: {MAX_ROUNDS}")

## 5. Test Server Connection

In [ ]:
import requests
import json

try:
    response = requests.get(f"{SERVER_URL}/health", timeout=10)
    print(f"✅ Server is healthy!")
    print(f"   Response: {response.json()}")
except requests.exceptions.ConnectionError:
    print(f"❌ Cannot connect to server at {SERVER_URL}")
    print(f"   Make sure the FL server is running on your VPS.")
    print(f"   Run on VPS: python -m server.grpc_server --host 0.0.0.0 --port 8080 --num-classes 4 --min-clients 4 --dp-enabled")
except Exception as e:
    print(f"❌ Error: {e}")

## 6. Run Federated Learning (4 Clients Sequential)

Each client trains locally on its data partition, then sends the update to the server.
After all 4 clients submit updates, the server aggregates them.

In [ ]:
import sys
sys.path.insert(0, ".")

from client.fed_client import FederatedClient, ClientConfig
from xai.explainer import XAIReportGenerator
import time

def create_client(client_id: int) -> FederatedClient:
    """Create an FL client with the specified ID."""
    config = ClientConfig(
        client_id=f"client_{client_id}",
        server_url=SERVER_URL,
        data_config=f"./data/client_{client_id}/data.yaml",
        model_variant=MODEL_VARIANT,
        num_classes=NUM_CLASSES,
        local_epochs=LOCAL_EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        imgsz=IMG_SIZE,
        dp_enabled=DP_ENABLED,
        dp_epsilon=DP_EPSILON,
        dp_delta=DP_DELTA,
        dp_max_grad_norm=DP_MAX_GRAD_NORM,
        save_dir=f"./results/client_{client_id}"
    )
    return FederatedClient(config)

# Create all 4 clients
clients = [create_client(i) for i in range(1, NUM_CLIENTS + 1)]
print(f"Created {len(clients)} FL clients")
for c in clients:
    print(f"  - {c.client_id}: data={c.config.data_config}")

In [ ]:
# Register all clients with the server
for client in clients:
    success = client.register_with_server()
    if success:
        print(f"✅ {client.client_id} registered successfully")
    else:
        print(f"❌ {client.client_id} registration failed")

In [ ]:
# Run Federated Learning - Sequential training of all 4 clients per round
import logging
logging.basicConfig(level=logging.INFO)

round_results = []

for round_num in range(1, MAX_ROUNDS + 1):
    print(f"\n{'='*60}")
    print(f"FEDERATED LEARNING ROUND {round_num}/{MAX_ROUNDS}")
    print(f"{'='*60}")
    
    round_start = time.time()
    round_metrics = []
    
    for client in clients:
        print(f"\n--- {client.client_id} ---")
        result = client.run_federated_round()
        
        if result["status"] == "success":
            print(f"  ✅ Done in {result['duration_seconds']:.1f}s")
            if "metrics" in result and result["metrics"]:
                print(f"  Metrics: {result['metrics']}")
            round_metrics.append(result)
        else:
            print(f"  ❌ Failed: {result.get('message', 'unknown error')}")
    
    round_duration = time.time() - round_start
    print(f"\n⏱️  Round {round_num} complete in {round_duration:.1f}s")
    print(f"   Successful clients: {len(round_metrics)}/{NUM_CLIENTS}")
    
    round_results.append({
        "round": round_num,
        "duration": round_duration,
        "successful_clients": len(round_metrics)
    })
    
    # Check server status
    try:
        status = requests.get(f"{SERVER_URL}/status", timeout=10).json()
        if status.get("current_round", 0) >= status.get("total_rounds", float("inf")):
            print("\n🏁 Training complete (server reached max rounds)!")
            break
    except:
        pass

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE - {len(round_results)} rounds")
print(f"{'='*60}")

## 7. Check Results & Privacy Report

In [ ]:
# Get server status
try:
    status = requests.get(f"{SERVER_URL}/status", timeout=10).json()
    print("📊 Server Status:")
    print(json.dumps(status, indent=2))
except Exception as e:
    print(f"Error getting status: {e}")

In [ ]:
# Get privacy report
try:
    privacy = requests.get(f"{SERVER_URL}/privacy_report", timeout=10).json()
    print("🔒 Privacy Report:")
    print(json.dumps(privacy, indent=2))
except Exception as e:
    print(f"Error getting privacy report: {e}")

In [ ]:
# Get round history
try:
    history = requests.get(f"{SERVER_URL}/round_history", timeout=10).json()
    print(f"📈 Round History ({len(history.get('rounds', []))} rounds):")
    for r in history.get("rounds", [])[-5:]:  # Show last 5 rounds
        print(f"  Round {r['round']}: clients={len(r['clients'])}, "
              f"duration={r['duration']:.1f}s, "
              f"epsilon_spent={r['privacy_cost'].get('epsilon', 0):.4f}")
except Exception as e:
    print(f"Error getting history: {e}")

## 8. Explainable AI (XAI) - Generate Explanations

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from xai.explainer import PrivacyAwareExplainer, XAIReportGenerator

# Use client 1's model for XAI explanation
client_model = clients[0].model

# Find a sample image
sample_dir = "./data/client_1/images/val"
if os.path.exists(sample_dir):
    sample_images = [f for f in os.listdir(sample_dir) if f.endswith(('.jpg', '.png'))]
    if sample_images:
        sample_path = os.path.join(sample_dir, sample_images[0])
        print(f"Using sample image: {sample_path}")
        
        # Load image
        image = cv2.imread(sample_path)
        print(f"Image shape: {image.shape}")
        
        # Generate XAI report
        report_gen = XAIReportGenerator(client_model, privacy_epsilon=1.0)
        report = report_gen.generate_report(
            image=image,
            target_class=None,  # Auto-detect top prediction
            output_dir="./xai_reports"
        )
        
        print("\n🧠 XAI Report Generated:")
        print(json.dumps({k: v for k, v in report.items() if k != 'explanations'}, indent=2, default=str))
        
        report_gen.cleanup()
    else:
        print("No validation images found")
else:
    print(f"Validation directory not found: {sample_dir}")

In [ ]:
# Visualize Grad-CAM explanation
if os.path.exists("./xai_reports/grad_cam_overlay.jpg"):
    grad_cam_img = cv2.imread("./xai_reports/grad_cam_overlay.jpg")
    grad_cam_img = cv2.cvtColor(grad_cam_img, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title("Original Image")
    plt.axis("off")
    
    plt.subplot(1, 2, 2)
    plt.imshow(grad_cam_img)
    plt.title("Grad-CAM Explanation (Privacy-Aware)")
    plt.axis("off")
    
    plt.tight_layout()
    plt.savefig("./xai_reports/comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Grad-CAM visualization saved to ./xai_reports/comparison.png")
else:
    print("Grad-CAM overlay not generated (model may need more training)")

## 9. Training Metrics Visualization

In [ ]:
import matplotlib.pyplot as plt

if round_results:
    rounds = [r["round"] for r in round_results]
    durations = [r["duration"] for r in round_results]
    successful = [r["successful_clients"] for r in round_results]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Round duration
    axes[0].plot(rounds, durations, 'b-o', markersize=3)
    axes[0].set_xlabel("Round")
    axes[0].set_ylabel("Duration (seconds)")
    axes[0].set_title("Training Duration per Round")
    axes[0].grid(True, alpha=0.3)
    
    # Successful clients
    axes[1].bar(rounds, successful, color='green', alpha=0.7)
    axes[1].set_xlabel("Round")
    axes[1].set_ylabel("Successful Clients")
    axes[1].set_title("Successful Clients per Round")
    axes[1].set_ylim(0, NUM_CLIENTS + 1)
    axes[1].axhline(y=NUM_CLIENTS, color='r', linestyle='--', label=f'Target ({NUM_CLIENTS})')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig("./results/training_metrics.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No training results to plot yet.")

## 10. Run Individual Client (Alternative)

If you want to run clients in **separate Colab notebooks** (e.g., for true parallel FL):

In [ ]:
# Run a single client (change CLIENT_ID for each Colab instance)
# Uncomment to use:

# CLIENT_ID = 1  # Change this: 1, 2, 3, or 4

# !python -m client.fed_client \
#   --client-id client_{CLIENT_ID} \
#   --server-url {SERVER_URL} \
#   --data-config ./data/client_{CLIENT_ID}/data.yaml \
#   --model yolo11n.pt \
#   --num-classes 4 \
#   --local-epochs 5 \
#   --batch-size 16 \
#   --lr 0.01 \
#   --dp-enabled \
#   --dp-epsilon 1.0 \
#   --dp-max-grad-norm 1.0 \
#   --max-rounds 100

---

## Notes

### VPS Server Setup
On your VPS (4 vCPU, 8GB RAM), run:
```bash
# Option 1: Docker
docker build -t fedx-server -f docker/Dockerfile.server .
docker run -d -p 8080:8080 --name fl-server fedx-server

# Option 2: Direct Python
pip install flask torch numpy pyyaml ultralytics
python -m server.grpc_server --host 0.0.0.0 --port 8080 --num-classes 4 --min-clients 4 --dp-enabled --dp-epsilon 1.0
```

### Dataset Classes (4)
| Class ID | Name | Description |
|----------|------|-------------|
| 0 | unripe | Green, not ready |
| 1 | underripe | Partially ripe |
| 2 | ripe | Ready for harvest |
| 3 | overripe | Past optimal |

### Privacy Guarantee
- **Local DP (ε=1.0)**: Each client clips gradients + adds Gaussian noise
- **Central DP**: Server adds additional noise after aggregation
- Training automatically stops when privacy budget is exhausted